<a href="https://colab.research.google.com/github/mulpuruvaishnavi-stack/spring_camp/blob/main/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import numpy as np

In [47]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
# Label Encoding- bcs computers don't understand words, we need to give numbers.
label_map = {
    'Wine': 0,
    'Beer': 1,
    'Whiskey': 2
}

# Splitting into X and y
X = []
y = []

for row in data:
    X.append(row[:3])
    y.append(label_map[row[3]])

# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

print("X(Features):")
print(X)

print("\ny(Labels):")
print(y)

X(Features):
[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]]

y(Labels):
[0 1 2 0 1 2 0 1]


In [48]:
def gini_impurity(y):
    # get unique classes and their counts
    classes, counts = np.unique(y, return_counts=True)
    # classes = [0, 1, 2]
    # counts  = [3, 3, 2]    #because gini needs probabilities, not raw labels

    # calculate probabilities
    probabilities = counts / len(y)

    # write gini impurity formula
    gini = 1 - np.sum(probabilities ** 2)

    return gini

gini = gini_impurity(y)
print("Gini Impurity of dataset:", gini)

def best_split(X, y):
    best_gini = float('inf') #we are starting from infinity since we need the least gini value
    best_feature = None      #initialising with none
    best_threshold = None

    n_samples, n_features = X.shape

    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])  # possible split points

        for threshold in thresholds:
            # Split data
            left_indices = X[:, feature] <= threshold
            right_indices = X[:, feature] > threshold

            y_left = y[left_indices]
            y_right = y[right_indices]

            # Skip invalid splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            # Compute Gini
            gini_left = gini_impurity(y_left)
            gini_right = gini_impurity(y_right)

            # Weighted Gini
            n = len(y)
            n_left = len(y_left)
            n_right = len(y_right)

            weighted_gini = (n_left/n) * gini_left + (n_right/n) * gini_right

            # Update best split
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


Gini Impurity of dataset: 0.65625


In [49]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def majority_class(y):
    classes, counts = np.unique(y, return_counts=True)
    return classes[np.argmax(counts)]

def build_tree(X, y, depth=0, max_depth=3, min_samples_split=2):

    # Stopping conditions - in recursion
    if len(np.unique(y)) == 1:
        return Node(value=y[0])

    if depth >= max_depth:
        return Node(value=majority_class(y))

    if len(y) < min_samples_split:
        return Node(value=majority_class(y))

    feature, threshold = best_split(X, y)

    if feature is None:
        return Node(value=majority_class(y))

    left_indices = X[:, feature] <= threshold
    right_indices = X[:, feature] > threshold

    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    left_child = build_tree(X_left, y_left, depth + 1, max_depth, min_samples_split)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth, min_samples_split)

    return Node(feature_index=feature, threshold=threshold, left=left_child, right=right_child)


# Example: Build Tree
tree = build_tree(X, y)

In [50]:
def predict_one(node, x):
    if node.value is not None:   # leaf node
        return node.value

    if x[node.feature_index] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(tree, X_test):
    predictions = []
    for x in X_test:
        predictions.append(predict_one(tree, x))
    return np.array(predictions)

test_data = np.array([
    [6.0, 2.1, 0],    # Expected: Beer
    [39.0, 0.05, 1],  # Expected: Whiskey
    [13.0, 1.3, 1]    # Expected: Wine
])

# Expected labels (encoded)
y_expected = np.array([1, 2, 0])  # Beer=1, Whiskey=2, Wine=0

predictions = predict(tree, test_data)

print("Predicted (encoded):", predictions)
print("Expected  (encoded):", y_expected)

accuracy = np.sum(predictions == y_expected) / len(y_expected)
print("Accuracy:", accuracy)

Predicted (encoded): [0 2 0]
Expected  (encoded): [1 2 0]
Accuracy: 0.6666666666666666
